<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Tensors and shapes

**[Tensors and shapes](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/)** · Deep Learning and Transformers Explained · Module 2, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read the shape of a tensor before you run code: scalars, vectors, matrices and 3-D tensors, the batch dimension, matrix products, broadcasting, the dtype and the device.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | A browser and a free Colab or Kaggle account. PyTorch is already installed there; the first cell checks it. No data download. |
| **You should know** | A matrix and the shape rule of matrix multiplication, from the mathematics course ([Vector operations](https://learning.nextia-ai.com/courses/math/m02/vector-operations/)). A layer as many neurons, from [Neurons and layers](https://learning.nextia-ai.com/courses/deep-learning/m01/neurons-and-layers/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M02-L01-tensors-and-shapes/tensors-and-shapes-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Four tensors and their shapes

A **tensor** is a box of numbers with a **shape**: one count for each dimension. `t.ndim` is the number of dimensions, `t.shape` the shape, and `t.numel()` the count of numbers.

The three tickets are the ones from the mathematics course: length, refund words, past tickets and a photo flag.

> **Predict.** What is the shape of `words = torch.zeros(2, 3, 4)`, and how many numbers does it hold? Then run the cell.

In [ ]:
score = torch.tensor(5.5)                      # one number
ticket = torch.tensor([2.0, 2.0, 0.0, 1.0])    # one ticket: 4 features
batch = torch.tensor([[2.0, 2.0, 0.0, 1.0],
                      [1.0, 0.0, 3.0, 0.0],
                      [3.0, 1.0, 1.0, 1.0]])   # 3 tickets, 4 features each
words = torch.zeros(2, 3, 4)                   # 2 tickets, 3 words, 4 numbers per word

for name, t in [("score", score), ("ticket", ticket), ("batch", batch), ("words", words)]:
    print(f"{name:<7} ndim {t.ndim}   shape {tuple(t.shape)}   numbers {t.numel()}")

> **Check.** You should see the shapes `()`, `(4,)`, `(3, 4)` and `(2, 3, 4)`, with 1, 4, 12 and 24 numbers.

The same 12 numbers can be cut into rows in another way, and nothing warns you. `reshape()` cuts a tensor into a new shape:

In [ ]:
flat = batch.flatten()          # the 12 numbers in one row
print(flat.reshape(4, 3))       # 4 "tickets" that do not exist

> **Check.** The second row is `[1., 1., 0.]`: the photo flag of ticket 1 and the first two features of ticket 2. The shape gives the numbers their meaning.

## 2. The batch dimension

The first dimension of `batch` counts the tickets: it is the **batch dimension**. The second holds the **features**.

> **Predict.** What is the shape of `batch[0]`? And of `batch[0:1]`? Then run the cell.

In [ ]:
print(batch[0], batch[0].shape)
print(batch[0:1], batch[0:1].shape)

> **Check.** You should see `torch.Size([4])` and `torch.Size([1, 4])`. An index drops the batch dimension; a slice keeps it.

## 3. Matrix products

`@` multiplies matrices. The inner numbers must be equal; the outer numbers give the result. Run the cell.

In [ ]:
w = torch.tensor([0.5, 2.0, 1.0, 0.5])   # one weight per feature
scores = batch @ w
print(scores, scores.shape)

> **Check.** You should see `tensor([5.5000, 3.5000, 5.0000]) torch.Size([3])`. By hand: 2 × 0.5 + 2 × 2 + 0 × 1 + 1 × 0.5 = 5.5.

Tomás writes the team weights with one row per team, (5, 4). The cell catches the error and prints PyTorch's message, so that the notebook can go on.

> **Predict.** Which product works: (3, 4) @ (4, 5) or (3, 4) @ (5, 4)? Then run the cell.

In [ ]:
W = torch.zeros(4, 5)          # 4 features in, 5 teams out
print((batch @ W).shape)
W_wrong = torch.zeros(5, 4)    # one row per team
try:
    print((batch @ W_wrong).shape)
except RuntimeError as error:  # print the message, so the notebook can go on
    print("RuntimeError:", error)
print((batch @ W_wrong.T).shape)   # .T turns (5, 4) into (4, 5)

> **Check.** You should see `torch.Size([3, 5])`, then `RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x4 and 5x4)`, then `torch.Size([3, 5])` again.

## 4. Broadcasting

**Broadcasting** compares two shapes from the right. Two sizes fit when they are equal, when one is 1, or when one is missing; a 1 or a missing size is stretched. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/) has a diagram and an explorer of the rule.

> **Predict.** Three cells follow: `batch - mean` with mean (4,), `batch / totals` with totals (3, 1), and `batch / totals_flat` with totals_flat (3,). Which one raises an error? Then run them.

In [ ]:
mean = batch.mean(dim=0)     # the mean of each column: shape (4,)
print(mean)
print(batch - mean)          # (3, 4) - (4,)

In [ ]:
totals = batch.sum(dim=1, keepdim=True)   # one total per ticket: shape (3, 1)
print(totals)
print(batch / totals)                     # (3, 4) / (3, 1)

In [ ]:
totals_flat = batch.sum(dim=1)            # shape (3,): keepdim forgotten
print(totals_flat.shape)
try:
    print(batch / totals_flat)
except RuntimeError as error:  # print the message, so the notebook can go on
    print("RuntimeError:", error)

> **Check.** Only the third cell raises an error: `The size of tensor a (4) must match the size of tensor b (3) at non-singleton dimension 1`. In the second cell, each row adds up to 1.

> **Your turn.** Compute `centered`: each ticket minus the mean of each feature, shape (3, 4). Then run the check cell.

In [ ]:
centered = ...  # each ticket minus the mean of each feature

In [ ]:
expect_hash('centered', centered, '053a9a2636c68e98')

## 5. dtype, NumPy and the device

Each tensor has a **dtype**. Networks use `float32`. Whole numbers are `int64`, and NumPy makes decimals `float64`. A product does not mix types. Each tensor also has a **device**: here, always `cpu`.

> **Predict.** `counts = torch.tensor([2, 2, 0, 1])` holds whole numbers. What happens with `counts @ w`? Then run the cell.

In [ ]:
import numpy as np

counts = torch.tensor([2, 2, 0, 1])     # whole numbers
print(counts.dtype, w.dtype)
try:
    print(counts @ w)
except RuntimeError as error:  # print the message, so the notebook can go on
    print("RuntimeError:", error)
print(counts.float() @ w)

from_numpy = torch.from_numpy(np.array([2.0, 2.0, 0.0, 1.0]))
print(from_numpy.dtype, "->", from_numpy.float() @ w)
print((batch @ w).numpy(), batch.device)

> **Check.** You should see `torch.int64 torch.float32`, the error `dot : expected both vectors to have same dtype, but found Long and Float`, then `tensor(5.5000)`, `torch.float64 -> tensor(5.5000)` and `[5.5 3.5 5. ] cpu`.

## 6. The Larkfield tickets as tensors

In the lesson *The forward pass*, a batch of 8 tickets has the shape (8, 2864): 2,864 word counts for each ticket. The first layer turns 2,864 numbers into 64.

> **Predict.** What is the shape of (8, 2864) @ (2864, 64)? How many bytes does the (8, 2864) tensor take, with 4 bytes per number? Then run the cell.

In [ ]:
tickets = torch.zeros(8, 2864)       # 8 tickets, 2,864 word counts each
W1 = torch.zeros(2864, 64)           # 2,864 inputs, 64 outputs
print((tickets @ W1).shape)
print(tickets.element_size(), "bytes per number,", tickets.element_size() * tickets.numel(), "bytes in all")

> **Check.** You should see `torch.Size([8, 64])` and `4 bytes per number, 91648 bytes in all`.

## 7. Change one thing: five tickets

> **Your turn.** The batch now has 5 tickets: `five = torch.ones(5, 4)`. Write down the shapes of `five @ w`, `five - five.mean(dim=0)` and `five / five.sum(dim=1, keepdim=True)` first. Then put the three shapes, as tuples, in the list `shapes_five`, and run the check cell.

In [ ]:
five = torch.ones(5, 4)                       # 5 tickets, 4 features
shapes_five = ...  # [tuple((five @ w).shape), ...] for the three results

In [ ]:
expect('shapes_five', shapes_five, [(5,), (5, 4), (5, 4)])

Only the batch dimension changed. The code does not depend on how many tickets there are.

## 8. Failure case: the error that does not appear

Priya compares the three scores with Grace's expected scores 5, 4 and 5, and computes the mean absolute error. She expects about 0.3.

> **Predict.** `scores` has shape (3,) and `targets` has shape (3, 1). What shape does `scores - targets` have? Then run the cell.

In [ ]:
targets = torch.tensor([[5.0], [4.0], [5.0]])   # one column: shape (3, 1)
errors = scores - targets                       # (3,) - (3, 1)
print(errors.shape)
print("mean absolute error:", round(errors.abs().mean().item(), 3))

> **Check.** You should see `torch.Size([3, 3])` and `mean absolute error: 0.778`, and no error. Both tensors stretched: PyTorch subtracted every target from every score.

The fix makes the shapes equal first. `squeeze(1)` removes the dimension of size 1. Run the cell.

In [ ]:
targets = targets.squeeze(1)    # shape (3,)
errors = scores - targets
print(errors.shape, errors)
print("mean absolute error:", round(errors.abs().mean().item(), 3))

> **Check.** You should see `torch.Size([3]) tensor([ 0.5000, -0.5000,  0.0000])` and `mean absolute error: 0.333`.

## Next

You can read the shape of a tensor, keep the batch dimension, check a matrix product, and predict when broadcasting works, fails, or stretches by accident. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/) has the diagrams, the explorer and the knowledge checks that count toward your certificate. Next, you set up PyTorch and run a first program whose output you can compare, number by number.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Set up PyTorch](https://learning.nextia-ai.com/courses/deep-learning/m02/set-up-pytorch/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m02/tensors-and-shapes/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this notebook are made-up examples.